# 🧑 Age & Gender Estimation - Model Training Notebook
This notebook trains the multi-output CNN model on the UTKFace dataset using Google Colab GPU.

In [ ]:
# Step 1: Verify GPU Access
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", gpus)
!nvidia-smi


In [ ]:
# Step 2: Download UTKFace Dataset from Kaggle
!pip install -q kagglehub
import kagglehub
import os

print("Downloading UTKFace dataset...")
path = kagglehub.dataset_download("jangedoo/utkface-new")
print("Dataset downloaded to:", path)

# Search for folder containing images
data_dir = path
for root, dirs, files in os.walk(path):
    jpg_count = sum(1 for f in files if f.endswith('.jpg'))
    if jpg_count > 1000:
        data_dir = root
        break
print(f"Using image folder: {data_dir} ({len(os.listdir(data_dir))} items)")


In [ ]:
# Step 3: Define Model Architecture
from tensorflow.keras import layers, models

IMG_SIZE = 128

def build_model(img_size=IMG_SIZE):
    inputs = layers.Input(shape=(img_size, img_size, 3), name="face_input")
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    gender_branch = layers.Dense(64, activation="relu")(x)
    gender_branch = layers.Dropout(0.3)(gender_branch)
    gender_output = layers.Dense(1, activation="sigmoid", name="gender_output")(gender_branch)

    age_branch = layers.Dense(64, activation="relu")(x)
    age_branch = layers.Dropout(0.3)(age_branch)
    age_output = layers.Dense(1, activation="linear", name="age_output")(age_branch)

    return models.Model(inputs=inputs, outputs=[age_output, gender_output], name="age_gender_cnn")

model = build_model()
model.summary()


In [ ]:
# Step 4: Dataset Pipeline & Label Extraction
import re
import numpy as np
from sklearn.model_selection import train_test_split

FILENAME_PATTERN = re.compile(r"^(\d+)_(\d+)_")

def load_labels(data_dir):
    filepaths, ages, genders = [], [], []
    for fname in os.listdir(data_dir):
        match = FILENAME_PATTERN.match(fname)
        if not match:
            continue
        age, gender = int(match.group(1)), int(match.group(2))
        if age > 100:
            continue
        filepaths.append(os.path.join(data_dir, fname))
        ages.append(age)
        genders.append(gender)
    return filepaths, np.array(ages, dtype=np.float32), np.array(genders, dtype=np.float32)

filepaths, ages, genders = load_labels(data_dir)
print(f"Found {len(filepaths)} valid image samples.")

train_fp, val_fp, train_age, val_age, train_gender, val_gender = train_test_split(
    filepaths, ages, genders, test_size=0.15, random_state=42
)

def _decode_and_preprocess(filepath, age, gender):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, {"age_output": age, "gender_output": gender}

def make_dataset(filepaths, ages, genders, batch_size=64, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, ages, genders))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths))
    ds = ds.map(_decode_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_fp, train_age, train_gender, batch_size=64, shuffle=True)
val_ds = make_dataset(val_fp, val_age, val_gender, batch_size=64, shuffle=False)


In [ ]:
# Step 5: Compile & Train Model
MODEL_OUT_PATH = "age_gender_model.h5"

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={"age_output": "mae", "gender_output": "binary_crossentropy"},
    loss_weights={"age_output": 0.5, "gender_output": 1.0},
    metrics={"age_output": "mae", "gender_output": "accuracy"},
)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(MODEL_OUT_PATH, monitor="val_loss", save_best_only=True),
]

print("Starting training for 30 epochs...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

model.save(MODEL_OUT_PATH)
print(f"Successfully saved trained model to {MODEL_OUT_PATH}")


In [ ]:
# Step 6: Download the trained age_gender_model.h5 to your computer
from google.colab import files
files.download('age_gender_model.h5')
